# Simple LangChain Ticket Solver

This notebook teaches the important LangChain concepts using **one support-ticket problem**.

After setup, you only call:

```python
solve_ticket("My account is locked and I have an important meeting in 10 minutes.")
```

The function automatically runs this flow:

```text
Message
  ↓
Clean input                  → RunnableLambda
  ↓
Analyse 3 things together    → RunnableParallel
  ↓
Keep original message        → RunnablePassthrough
  ↓
Create reliable fields       → Structured Output
  ↓
Generate customer reply      → LCEL Sequence
  ↓
Final result
```

### Intentionally removed

To keep this lesson simple, this notebook does **not** include:

- RunnableBranch / Router
- Retry
- Fallback
- LangSmith
- Streaming
- Batch
- Async

The goal is simply: **one message → one complete flow → one final result.**


## Google Colab setup

Use a fresh **Python 3.12 or 3.13 CPU runtime** for each notebook. In the Secrets sidebar, add `OPENAI_API_KEY` and enable **Notebook access** for this notebook. Run cells from top to bottom. If installation requests a restart, restart the session and continue with the imports/credentials cell (do not reinstall repeatedly). No Drive mount or local project files are needed.

The default model is `gpt-4o-mini`. API billing/quota and model access are required; a ChatGPT subscription does not supply API quota. Keys are never printed. Package conflicts can occur if multiple framework notebooks share one runtime—start a fresh session when switching frameworks.


## 1. Install LangChain

Run this cell once.


In [ ]:
%pip -q install "langchain-core==1.6.9" "langchain-openai==1.7.0" "python-dotenv==1.2.4"


## 2. Add your OpenAI API key

The notebook automatically checks in this order:
1. **Local `.env` file** (in project root)
2. **Environment variable** (`OPENAI_API_KEY`)
3. **Google Colab Secrets** (click 🔑 Secrets on the left)
4. **Interactive prompt** (if none of the above are found)


In [ ]:
# os lets us read and set environment variables, where the API client expects its key.
import os

# Return one non-empty key. This only loads credentials; it does not call OpenAI.
def read_openai_key():
    # In Colab, always use this notebook's enabled secret.
    # This import succeeds in Colab. Local Jupyter/Python normally takes the fallback.
    try:
        from google.colab import userdata
    except ImportError:
        # A local Python/Jupyter session normally has no google.colab module.
        # None is a marker meaning "there is no Colab Secrets reader here";
        # it is not an API key and does not clear an existing environment key.
        # The `if userdata is not None` block below is therefore skipped.
        # Execution continues to the local fallback: environment/.env, then
        # a hidden prompt if no key was found. In Colab, the import succeeds
        # and userdata remains the module used to read the enabled secret.
        userdata = None
    if userdata is not None:
        try:
            # The secret name must match exactly, with Notebook access enabled.
            key = userdata.get("OPENAI_API_KEY")
        except Exception:
            # Give a useful setup message without exposing secret/error details.
            # In Colab, stop here rather than silently using a different local key.
            raise RuntimeError(
                "In Colab Secrets, add OPENAI_API_KEY and enable Notebook access, "
                "then rerun this cell."
            ) from None
        # Reject missing/blank values; strip removes accidental surrounding spaces.
        if not isinstance(key, str) or not key.strip():
            raise RuntimeError("The OPENAI_API_KEY Colab secret is empty.")
        return key.strip()

    # Local Python fallback; never overwrite an existing environment key.
    try:
        from dotenv import load_dotenv, find_dotenv
        # Search from the current folder upward for .env. Existing environment values win.
        load_dotenv(find_dotenv(usecwd=True), override=False)
    except ImportError:
        pass  # python-dotenv is optional here; environment and hidden input still work.
    # Read an existing environment key, or the key just loaded from .env.
    key = os.environ.get("OPENAI_API_KEY", "").strip()
    if not key:
        from getpass import getpass
        # getpass hides what you type; never paste a key directly into notebook code.
        key = getpass("Enter OPENAI_API_KEY (hidden): ").strip()
    if not key:
        raise ValueError("OPENAI_API_KEY cannot be empty.")
    return key

# Make the key available to ChatOpenAI. Do not print the environment value.
os.environ["OPENAI_API_KEY"] = read_openai_key()

# "Ready" means loaded; authentication/quota are checked on the first model request.
print("✅ OpenAI API key is ready.")


## 3. Build the ticket-solving flow

You only need to understand the comments.

Each LangChain feature solves one small problem:

- **Lambda** → clean normal Python input
- **Parallel** → do independent analyses together
- **Passthrough** → keep old data while adding new data
- **Structured Output** → force predictable fields
- **LCEL (`|`)** → connect the steps


In [ ]:
# Wraps the OpenAI chat API as a runnable that fits into an LCEL pipeline.
from langchain_openai import ChatOpenAI
# Replaces placeholders such as {ticket} with values from the input dictionary.
from langchain_core.prompts import ChatPromptTemplate
# Converts the model message into a plain string for the following step.
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnableLambda,       # Turns an ordinary Python function into a runnable step.
    RunnableParallel,     # Gives the same input to branches and collects named results.
    RunnablePassthrough,  # Keeps existing input fields; .assign() adds new fields.
)

# ---------------------------------------------------------
# MODEL
# ---------------------------------------------------------

# Creating the client configures the model; .invoke() later sends the actual request.
model = ChatOpenAI(
    model="gpt-4o-mini"
)


### Step 1: Input Validation & Cleaning (`clean_step`)
Use `RunnableLambda` to apply standard Python string validation and formatting before the ticket enters any LLM chain.


In [ ]:
# ---------------------------------------------------------
# 1. RUNNABLE LAMBDA: clean_step
# Problem: user input may contain spaces or may be empty.
# Solution: use normal Python logic inside LangChain.
# ---------------------------------------------------------

def clean_message(message: str) -> dict:
    message = message.strip()

    if not message:
        raise ValueError("Ticket cannot be empty.")

    return {"ticket": message}


# Later, clean_step.invoke(text) will call clean_message(text).
clean_step = RunnableLambda(clean_message)


### Step 2: Parallel Analysis & Preservation (`keep_ticket_and_add_analysis`)
Run `summary`, `category`, and `urgency` chains concurrently using `RunnableParallel`, then preserve the original message while attaching the analysis using `RunnablePassthrough.assign`.


In [ ]:
# ---------------------------------------------------------
# 2. RUNNABLE PARALLEL & RUNNABLE PASSTHROUGH: keep_ticket_and_add_analysis
# Problem: summary, category, and urgency are independent.
# Solution: run all three from the same ticket, then attach them.
# ---------------------------------------------------------

# The | operator passes each runnable's output into the next runnable:
# ticket dictionary -> formatted prompt -> model message -> plain text.
summary_chain = (
    ChatPromptTemplate.from_template(
        "Summarize this support ticket in one short sentence:\n{ticket}"
    )
    | model
    | StrOutputParser()
)

category_chain = (
    ChatPromptTemplate.from_template(
        "Return only one category: billing, technical, or account.\n"
        "Ticket: {ticket}"
    )
    | model
    | StrOutputParser()
)

urgency_chain = (
    ChatPromptTemplate.from_template(
        "Return only one urgency: low, medium, or high.\n"
        "Ticket: {ticket}"
    )
    | model
    | StrOutputParser()
)

# These three branches use the same ticket; they do not depend on each other.
# The result is a dictionary with summary, category and urgency.
analyse_ticket = RunnableParallel(
    summary=summary_chain,
    category=category_chain,
    urgency=urgency_chain,
)

# Keep the ticket and attach that analysis dictionary under the "analysis" key.
keep_ticket_and_add_analysis = RunnablePassthrough.assign(
    analysis=analyse_ticket
)


### Step 3: Decision Chain (`add_decision`)
Keep it very simple: a straightforward prompt that takes the ticket and analysis, and decides if human escalation is needed.


In [ ]:
# ---------------------------------------------------------
# 3. SIMPLE DECISION CHAIN: add_decision
# Problem: review the analysis and make a final decision.
# Solution: a simple prompt + model!
# ---------------------------------------------------------

decision_prompt = ChatPromptTemplate.from_template(
    """
Review this ticket and the initial analysis:

Ticket:
{ticket}

Analysis:
{analysis}

Give the final decision:
1. Should this ticket be escalated to a human? (Yes/No)
2. Brief reason why.
"""
)

decision_chain = (
    decision_prompt
    | model
    | StrOutputParser()
)

# Keep ticket + analysis, and ADD the final decision.
# decision_chain can read ticket and analysis; its string is stored as decision.
add_decision = RunnablePassthrough.assign(
    decision=decision_chain
)


### Step 4: Customer Reply (`add_reply`)
Generate a helpful and empathetic customer reply using the ticket and the decision.


In [ ]:
# ---------------------------------------------------------
# 4. CUSTOMER REPLY: add_reply
# LCEL Sequence for reply + RunnablePassthrough.assign
# ---------------------------------------------------------

reply_prompt = ChatPromptTemplate.from_template(
    """
You are a customer-support assistant.

Customer message:
{ticket}

Decision:
{decision}

Write a short and helpful reply to the customer.
Do not mention prompts, AI, or internal system details.
"""
)

reply_chain = (
    reply_prompt
    | model
    | StrOutputParser()
)

# Keep previous state and ADD customer reply
# Keep ticket, analysis and decision, then attach the customer reply string.
add_reply = RunnablePassthrough.assign(
    reply=reply_chain
)


### Step 5: Complete Pipeline (`ticket_workflow`)
Connect all 4 modular steps together into a single end-to-end pipeline with the pipe (`|`) operator!


In [ ]:
# ---------------------------------------------------------
# 5. COMPLETE WORKFLOW PIPELINE
# Connect all 4 steps together with the | operator
# ---------------------------------------------------------

ticket_workflow = (
    clean_step
    | keep_ticket_and_add_analysis
    | add_decision
    | add_reply
)


### Step 6: Runner Function (`solve_ticket`)
Invoke the entire workflow for any customer message and display a formatted summary.


In [ ]:
# ---------------------------------------------------------
# 6. THE ONLY FUNCTION YOU NEED TO CALL
# ---------------------------------------------------------

def solve_ticket(message: str):
    """
    Give this function one customer message.
    It runs the entire LangChain flow and prints the result.
    """

    # Start with the input text. The pipeline returns the accumulated result dictionary.
    result = ticket_workflow.invoke(message)

    print("\n========== TICKET RESULT ==========")
    print("Message  :", result["ticket"])
    print("\nAnalysis :")
    for key, val in result["analysis"].items():
        print(f"  - {key}: {val}")
    print("\nDecision :\n", result["decision"])
    print("\nReply to customer:\n", result["reply"])
    print("===================================")

    return result


print("✅ Ticket workflow is ready.")
print("Now call: solve_ticket('your message')")


## 4. Run ONE function

This is the only function you need to run for the actual ticket.

Change the sentence and run it again whenever you want.


In [ ]:
result = solve_ticket(
    "My account is locked and I have an important meeting in 10 minutes."
)


# What just happened?

When you ran:

```python
solve_ticket("My account is locked...")
```

the function automatically did this:

```text
"My account is locked..."
          ↓
1. RunnableLambda
   Clean the message
          ↓
2. RunnableParallel
   ├── Summary
   ├── Category
   └── Urgency
          ↓
3. RunnablePassthrough
   Keep original message + add analysis
          ↓
4. Structured Output
   Create:
   category
   urgency
   summary
   escalate
          ↓
5. LCEL reply chain
   Generate customer response
          ↓
FINAL RESULT
```

## The Runnable patterns in this project

```text
RunnableSequence     → created by A | B | C
RunnableParallel     → run independent analysis together
RunnableLambda       → use normal Python inside the flow
RunnablePassthrough  → keep old data while adding new data
```

That is the entire lesson.

**Input one ticket → LangChain runs the flow → get the final decision and reply.**
